# Experiment with various scikit learn models


In [ ]:
from datetime import datetime
import os
import random
from IPython.display import display, Image, HTML

import pandas as pd
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tempfile


from lime.lime_text import LimeTextExplainer
from scipy.special import expit, softmax
import shap
import mlflow

import json
from typing import Tuple, Protocol, Any
import mlflow.sklearn
from mlflow.entities import Run
import tensorflow as tf
from pathlib import Path
import joblib
from scipy.stats import bootstrap
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.model_selection import  RandomizedSearchCV
from sklearn.svm import SVC, LinearSVC
from sklearn.linear_model import SGDClassifier
from sklearn.experimental import enable_halving_search_cv
from sklearn.preprocessing import LabelEncoder, Normalizer
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
)

from scipy.stats import loguniform, t
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    HalvingRandomSearchCV,
)
from sklearn.multiclass import OneVsRestClassifier
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, PassiveAggressiveClassifier
from sklearn.linear_model import RidgeClassifier
from sentence_transformers import SentenceTransformer

from typing import Iterable
from sklearn.base import BaseEstimator, TransformerMixin

from ixbrl_ai.display import heading, display_wide
from ixbrl_ai.sample import DataSample

# 1. Config, setup mlflow, load data

In [ ]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# Set random seed
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

In [ ]:
experiment_name = "model-compare"
mlflow.set_experiment(experiment_name)
mlflow.sklearn.autolog()

In [ ]:
dataset_version = "v13"
dataset_name = f"data/canonicalized_split_{dataset_version}.parquet"
dataset_pl = pl.read_parquet(dataset_name)
X = "canonical_description"
y = "label"

For consistency over all the models, use encoded label. But for most scikit learn models I could use canonical label(text) directly.  

In [ ]:
rng = np.random.default_rng(seed=SEED)
idx_holdout_10k = rng.choice(dataset_pl.filter(pl.col("holdout")).select("row_id"), size=10000, replace=False)
holdout_10k_pl = dataset_pl.filter(pl.col("row_id").is_in(idx_holdout_10k.flatten()))
dataset_pl = dataset_pl.with_columns(pl.col("row_id").is_in(idx_holdout_10k.flatten()).alias("holdout_10k"))


# 2. Define Functions

In [ ]:
def get_split(dataset_pl: pl.DataFrame, subset: DataSample) -> Tuple[pl.DataFrame, pl.DataFrame, pl.DataFrame]:
    """Filters to train, test, and holdout splits

    Args:
        dataset_df (pl.DataFrame): Dataset
        subset (DataSample): A subset type which is a sample or full dataset

    Returns:
        Tuple[pl.DataFrame, pl.DataFrame, pl.DataFrame]: test, train and holdout
    """

    return (
        dataset_pl.filter(pl.col(subset.label), pl.col("split") == "train"),
        dataset_pl.filter(pl.col("split") == "test"),
        dataset_pl.filter(pl.col("split") == "holdout"),
    )


The dataset is very big and can take a long time to train or be too large to practically train some models.   
The test and holdout keep original statified split and population size and remain unchanged regarless of the training dataset.

In [ ]:
def run_grid_search(
    grid_search: GridSearchCV | RandomizedSearchCV | HalvingRandomSearchCV,
    dataset_name: str,
    dataset_pl: pl.DataFrame,
    subset: DataSample,
    run_name: str,
    save_grid: bool = True,
) -> GridSearchCV | RandomizedSearchCV | HalvingRandomSearchCV:
    """Runs grid_search or random search using ml flow, to record details and tests best model against test_split

    Args:
        grid_search (GridSearchCV): Grid Search
        dataset_name (str): Name of dataset
        dataset_pl (pl.DataFrame): Full Dataset
        subset (DataSample): Subset to use
        run_name (str): Name for the run
        save_grid (bool): Whether to save the grid search object as an artifact. This can be large, and cause crashing so optional. Defaults to True.
    """
    start_time = datetime.now()
    print(start_time)

    train_pl, test_pl, _ = get_split(dataset_pl, subset=subset)
    with mlflow.start_run(run_name=run_name):
        mlflow.set_tag("subset", subset.label)
        mlflow.set_tag("dataset", dataset_name)

        # Converty to mupy so mlflow logging works
        grid_search.fit(train_pl[X].to_numpy(), train_pl[y].to_numpy())
        y_test_pred = grid_search.best_estimator_.predict(test_pl[X].to_numpy())

        mlflow.log_metric("test_accuracy", accuracy_score(test_pl[y], y_test_pred))
        mlflow.log_metric(
            "test_f1_weighted", f1_score(test_pl[y], y_test_pred, average="weighted")
        )
        mlflow.log_metric(
            "test_f1_macro", f1_score(test_pl[y], y_test_pred, average="macro")
        )
        mlflow.log_metric(
            "test_precision_macro",
            precision_score(test_pl[y], y_test_pred, average="macro"),
        )
        mlflow.log_metric(
            "test_recall_macro", recall_score(test_pl[y], y_test_pred, average="macro")
        )
        mlflow.sklearn.log_model(grid_search.best_estimator_, name="best_model")
        cv_results_df = pd.DataFrame(grid_search.cv_results_)

        # Need to manually save since not working with Halving
        with tempfile.TemporaryDirectory() as d:
            d = Path(d)
            csv_path = Path(d, "cv_results.csv")
            cv_results_df.to_csv(csv_path, index=False)
            mlflow.log_artifact(str(csv_path), artifact_path="cv_results")

            params_path = Path(d, "best_params.json")
            params_path.write_text(
                json.dumps(grid_search.best_params_, indent=2, default=str)
            )
            mlflow.log_artifact(str(params_path), artifact_path="cv_results")

            search_path = Path(d, "grid_search.joblib")
            if save_grid:
                joblib.dump(grid_search, search_path)
                mlflow.log_artifact(str(search_path), artifact_path="search")

    end_time = datetime.now()
    print(f"Finished at: {end_time} duration: {end_time - start_time}")

    return grid_search


def load_ml_run(
    run_id: str,
    load_grid_search: bool = True,
) -> Tuple[
    Run,
    GridSearchCV | RandomizedSearchCV | HalvingRandomSearchCV | None,
    pl.DataFrame,
    list[Path],
]:
    """Returns the artifacts for the mlflow run

    Args:
        run_id (str): Run id

    Returns:
        Tuple[Run, GridSearchCV | RandomizedSearchCV | HalvingRandomSearchCV | None, pl.DataFrame, list[Path]]: Run details, grid search, gridsearch cv results, image paths
    """
    run = mlflow.get_run(run_id)
    local_dir = mlflow.artifacts.download_artifacts(run_id=run_id)
    local_dir = Path(local_dir)

    joblib_paths = list(local_dir.rglob("*.joblib"))
    csv_paths = list(local_dir.rglob("*.csv"))
    image_paths = list(local_dir.rglob("*.png"))

    if not joblib_paths:
        grid_search = None
    else:
        grid_search = joblib.load(joblib_paths[0])

    cv_results_pl = pl.read_csv(csv_paths[0])

    return run, grid_search, cv_results_pl, image_paths


def load_ml_flow(
    experiment_name: str, run_name: str, index: int = 0, load_grid_search: bool = True
) -> Tuple[
    Run,
    GridSearchCV | RandomizedSearchCV | HalvingRandomSearchCV | None,
    pl.DataFrame,
    list[Path],
]:
    """Returns artifacts for the mlflow name

    Args:
        experiment_name (str): _description_
        run_name (str): _description_
        index (int, optional): _description_. Defaults to 0.

    Returns:
        Tuple[Run, GridSearchCV | RandomizedSearchCV | HalvingRandomSearchCV | None, pl.DataFrame, list[Path]]: _description_
    """

    exp = mlflow.get_experiment_by_name(experiment_name)
    runs = mlflow.search_runs(
        experiment_ids=[exp.experiment_id],
        filter_string=(
            f"tags.mlflow.runName = '{run_name}' AND attributes.status = 'FINISHED'"
        ),
        order_by=["attributes.start_time DESC"],
    )
    print(index)
    run_id = runs.loc[index, "run_id"]

    return load_ml_run(run_id, load_grid_search=load_grid_search)

def add_confidence_interval(
    cv_results_pl: pl.DataFrame, confidence: float = 0.95, metric: str = "f1_macro"
) -> pl.DataFrame:
    """Adds margin of error to cv results

    Args:
        cv_results_pl (pl.DataFrame): CV results
        confidence (float, optional): Confidence level. Defaults to 0.95.

    Returns:
        pl.DataFrame: CV results with confidence interval
    """

    if f"split0_test_{metric}" in cv_results_pl.columns:
        scores = cv_results_pl.select(pl.col(rf"^split[0-9]*_test_{metric}$"))
    else:
        scores = cv_results_pl.select(pl.col(r"^split[0-9]*_test_score$"))

    k = len(scores.columns)
    stds = scores.to_pandas().std(axis=1, ddof=1)

    t_crit = t.ppf((1 + confidence) / 2, df=k - 1)
    margin_of_error = t_crit * (stds / np.sqrt(k))

    return cv_results_pl.with_columns(
        pl.Series(margin_of_error).alias(f"ci_{metric}_margin_of_error"),
    )


def compare_to_top(
    cv_results_pl: pl.DataFrame,
    absolute_difference: float = 0.01,
    confidence: float = 0.95,
) -> pl.DataFrame:
    """Compares to top both in absolute terms and using t test
    The t test might is only really valid if there are lots of folds, so should just be informative rather than definitive

    Args:
        cv_results_pl (pl.DataFrame): Dataframe
        absolute_difference (float, optional): Absolute difference for something to be considered as close enough. Defaults to 0.01.
        confidence (float, optional): Confidence interval for t test for it to be worse than. Defaults to 0.95.

    Returns:
        pl.DataFrame: _description_
    """


    if "split0_test_f1_macro" in cv_results_pl.columns:
        rank_expr = pl.col(r"^rank_test_f1_macro$")
        test_score_exp = pl.col(r"^split[0-9]*_test_f1_macro$")
        mean_expr = pl.col(r"^mean_test_f1_macro$")
    else:
        rank_expr = pl.col(r"^rank_test_score$")   
        test_score_exp = pl.col(r"^split[0-9]*_test_score$")
        mean_expr = pl.col(r"^mean_test_score$")

    top_scores = (
        cv_results_pl.filter(rank_expr == 1)
        .select(test_score_exp)
        .to_numpy()[0, :]
    )

    k = len(top_scores)
    differences = cv_results_pl.select(test_score_exp).to_pandas() - top_scores
    means = differences.mean(axis=1)
    stds = differences.std(axis=1, ddof=1)

    t_values = means / (stds / np.sqrt(k))
    t_values = t_values.fillna(0)

    t_crit = t.ppf(1 - confidence, df=k - 1)
    mean_test_difference = mean_expr.max() - mean_expr

    return cv_results_pl.with_columns(
        mean_test_difference.alias("mean_test_difference"),
        mean_test_difference.lt(absolute_difference).alias(f"mean_test_to_top"),
        pl.Series(t_values).alias("t_values"),
        pl.Series(t_values >= t_crit).alias("t_not_significantly_worse"),
    )


def display_good_params(cv_results_pl: pl.DataFrame) -> None:
    """Shows params that aren't proven to be worse, split by model type

    Args:
        cv_results_pl (pl.DataFrame): DataFrame
    """
    for group in cv_results_pl["param_model"].unique():
        display(f"Group: {group}")
        (
            cv_results_pl.select(
                "param_model",
                "params",
                "mean_test_score",
                "mean_fit_time",
                "rank_test_score",
                "mean_test_difference",
                "mean_test_to_top",
                "t_values",
                "t_not_significantly_worse",
            )
            .filter(pl.col("param_model") == group, pl.col("t_not_significantly_worse"))
            .pipe(display_wide, rows=1000)
        )


def display_group_params(cv_results_pl: pl.DataFrame) -> None:
    """Shows params that aren't proven to be worse, split by model type

    Args:
        cv_results_pl (pl.DataFrame): DataFrame
    """
    for group in cv_results_pl["param_model"].unique():
        display(f"Group: {group}")
        (
            cv_results_pl.select(
                "param_model",
                "params",
                "mean_test_score",
                "mean_fit_time",
                "rank_test_score",
                "mean_test_difference",
                "mean_test_to_top",
                "t_values",
                "t_not_significantly_worse",
            )
            .filter(pl.col("param_model") == group)
            .sort("rank_test_score", descending=False)
            .pipe(display_wide, rows=100)
        )




def get_metrics(
    grid_search: GridSearchCV, dataset_pl: pl.DataFrame, test_field: str, n_resamples: int = 1000
) -> dict[str, float]:
    """Displays and gets macro metrics

    Args:
        grid_search (GridSearchCV): Gridsearch
        dataset_pl (pl.DataFrame): Dataset
        test_field (str): Field name of the column to filter dataset by

    Returns:
        dict[str, float]: Metrics
    """
    testing_pl = dataset_pl.filter(pl.col(test_field))
    predictions = grid_search.predict(testing_pl[X])
    precision, recall, f1, support = precision_recall_fscore_support(
        testing_pl[y], predictions, average="macro"
    )
    
    metrics = {
        "accuracy": accuracy_score(testing_pl[y], predictions),
        "f1_macro": f1,
        "precision": precision,
        "recall": recall,
        "support": support,
    }
    for key, value in metrics.items():
        print(f"{key}: {value}")

    return metrics




def plot_scores_by_model(data_pl: pl.DataFrame) -> None:
    """Plots the scores by model

    Args:
        data_pl (pl.DataFrame): Dataset with scores and model details
    """
    sns.scatterplot(data=data_pl, x="param_model", y="mean_test_score")
    plt.title("F1 macro score for different model types")
    plt.xticks(rotation=90)
    plt.ylabel("F1 macro score")
    plt.xlabel("Model")
    plt.show()


def plot_scores_vs_training_time(data_pl: pl.DataFrame) -> None:
    """Plots scores vs training time

    Args:
        data_pl (pl.DataFrame): Dataset
    """
    sns.scatterplot(
        data=data_pl,
        x="mean_fit_time",
        y="mean_test_score",
        hue="param_model",
        style="param_model",
    )
    plt.title("F1 Macro Score vs Training Time")
    plt.xticks(rotation=90)
    plt.ylabel("F1 Macro Score")
    plt.xlabel("Training Time")
    plt.legend(title="Model", bbox_to_anchor=(1.02, 1))
    plt.show()


class SentenceVectorizer(BaseEstimator, TransformerMixin):
    def __init__(
        self, model_name: str, prefix: str | None = None, batch_size: int = 64
    ):
        self.model_name = model_name
        self.prefix = prefix
        self.batch_size = batch_size
        self._model = None

    def fit(self, X: Iterable[str], y=None):
        self._model = SentenceTransformer(self.model_name)
        return self

    def transform(self, X: Iterable[str]) -> np.ndarray:
        texts = list(X)
        if self.prefix is not None:
            texts = [f"{self.prefix}{t}" for t in texts]
        embedded = self._model.encode(
            texts,
            batch_size=self.batch_size,
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False,
        )
        return embedded

    def __getstate__(self):
        state = self.__dict__.copy()
        state["_model"] = None
        return state

    def __setstate__(self, state):
        self.__dict__.update(state)
        self._model = None



class SupportsPredict(Protocol):
    def predict(self, X: Any) -> Any: ...

def bootstrap_ci_fast(dataset_pl: pl.DataFrame, model: SupportsPredict, test_field: str, n_bootstrap: int = 1000, ci: float = 0.95):
    """Calculate bootstrap confidence intervals for accuracy and F1 score.
    
    Args:
        dataset_pl (pl.DataFrame): The dataset as a Polars DataFrame.
        model (SupportsPredict): The trained model with a predict method.
        test_field (str): The column name in dataset_pl that indicates the test set.
        n_bootstrap (int): The number of bootstrap samples to generate.
        ci (float): The confidence level for the intervals (e.g., 0.95 for 95% confidence intervals)."""
    
    rng = np.random.default_rng(seed=SEED)
    test_pl = dataset_pl.filter(pl.col(test_field))
    n = test_pl.height
    y_true = test_pl[y].to_numpy()
    y_pred = model.predict(test_pl[X].to_numpy())
    idx = rng.integers(0, n, size=(n_bootstrap, n))
    acc_scores = (y_true[idx] == y_pred[idx]).mean(axis=1)
    accuracy = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro")
    f1_scores = np.array([f1_score(y_true[idx[i]], y_pred[idx[i]], average="macro") for i in range(n_bootstrap)])
    alpha = (1 - ci) / 2
    def ci_stats(scores, estimate):
        return {
            "mean": estimate,
            "lower": np.percentile(scores, alpha * 100),
            "upper": np.percentile(scores, (1 - alpha) * 100),
        }

    return {"accuracy": ci_stats(acc_scores, accuracy), "f1": ci_stats(f1_scores, f1)}

In [ ]:
dataset_pl

# 3. Compare different population sizes.
The full training population takes a very long time to train against and isn't possible to train all the different models and hyperparameters. 

Let's test the 1%, 10% and 100% populations against a few models and see if the scores on 1% are representative against the 100% training poplulation. 

In [ ]:

pipe = Pipeline(
    [
        (
            "tfidf",
            TfidfVectorizer(
                analyzer="word",
                ngram_range=(1, 2),
                lowercase=True,
                min_df=1,
                norm="l2",
                use_idf=True,
            ),
        ),
        ("model", DummyClassifier(random_state=SEED)),
    ]
)

param_grid = [
    {
        "model": [DummyClassifier()],
        "model__random_state": [SEED],
        "model__strategy": ["stratified", "most_frequent", "uniform"],
    },
    {
        "model": [LinearSVC()],
    },
    {
        "model": [SGDClassifier()],
        "model__random_state": [SEED],
    },
    {
        "model": [DecisionTreeClassifier()],
        "model__random_state": [SEED],
    },
    {
        "model": [RandomForestClassifier()],
        "model__random_state": [SEED],
    },
    {
        "model": [MultinomialNB()],
    },
        {
        "model": [ComplementNB()],
    },
    {
        "model": [RidgeClassifier()],
    },
    {
        "model": [PassiveAggressiveClassifier()],
        "model__random_state": [SEED],
    },
]


In [ ]:
subset = DataSample.sample_1_pct
run_name = f"compare_datasets_v4_{dataset_version}_{subset.label}"

In [ ]:
grid_search = GridSearchCV(
    estimator=pipe, 
    param_grid=param_grid, 
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED), 
    scoring="f1_macro", 
    n_jobs=-1,
    verbose=3)

grid_run = run_grid_search(grid_search=grid_search, dataset_name=dataset_name, dataset_pl=dataset_pl, subset=subset, run_name=run_name)



In [ ]:
run, grid_search, cv_results_pl, image_paths = load_ml_flow(experiment_name=experiment_name, run_name=run_name)
cv_results_pl = compare_to_top(cv_results_pl)
display_wide(cv_results_pl)
display_group_params(cv_results_pl)

Average fit time ranged from 0.003 to 22.6 seconds

In [ ]:
plot_scores_by_model(cv_results_pl)

In [ ]:
plot_scores_vs_training_time(cv_results_pl)

- The models that took longer to train generally had better F1 macro scores
- But there were a number that had good F1 macro scores and were quick to train. 

In [ ]:
subset = DataSample.sample_10_pct
run_name = f"compare_datasets_v4_{dataset_version}_{subset.label}"

In [ ]:

grid_search = GridSearchCV(
    estimator=pipe, 
    param_grid=param_grid, 
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED), 
    scoring="f1_macro", 
    n_jobs=-1,
    verbose=3)

run_grid_search(grid_search=grid_search, dataset_name=dataset_name, dataset_pl=dataset_pl, subset=subset, run_name=run_name)

In [ ]:
run_10, grid_search_10, cv_results_10_pl, image_paths_10 = load_ml_flow(experiment_name=experiment_name, run_name=run_name)
cv_results_10_pl = compare_to_top(cv_results_10_pl).pipe(add_confidence_interval, confidence=0.95)
display_wide(cv_results_10_pl)
display_group_params(cv_results_10_pl)

for path in image_paths_10:
    display(Image(path))

In [ ]:
plot_scores_by_model(cv_results_10_pl)

In [ ]:
cv_results_pl

In [ ]:
subset = DataSample.sample_100_pct
run_name = f"compare_datasets_v5_{dataset_version}_{subset.label}"

In [ ]:
grid_search = GridSearchCV(
    estimator=pipe, 
    param_grid=param_grid, 
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED), 
    scoring="f1_macro", 
    n_jobs=-1,
    verbose=3)

run_grid_search(grid_search=grid_search, dataset_name=dataset_name, dataset_pl=dataset_pl, subset=subset, run_name=run_name)

In [ ]:
run_100, grid_search_100, cv_results_100_pl, image_paths_100 = load_ml_flow(experiment_name=experiment_name, run_name=run_name)
cv_results_100_pl = (
    cv_results_100_pl
    .pipe(compare_to_top)
    .pipe(add_confidence_interval, confidence=0.95)
)
display_wide(cv_results_100_pl)
display_group_params(cv_results_100_pl)


In [ ]:
cv_renamed_1_pl = cv_results_pl.select(
    "param_model", "params", "mean_test_score", "rank_test_score", "mean_fit_time", "t_values", "t_not_significantly_worse"
).rename(
    {
        "rank_test_score": "rank_test_score_1",
        "mean_fit_time": "mean_fit_time_1",
        "mean_test_score": "mean_test_score_1",
        "t_not_significantly_worse": "t_not_significantly_worse_1",
        "t_values": "t_values_1",
    }
)
cv_renamed_10_pl = cv_results_10_pl.select(
    "param_model", "params", "mean_test_score", "rank_test_score", "mean_fit_time", "t_values", "t_not_significantly_worse"
).rename(
    {
        "rank_test_score": "rank_test_score_10",
        "mean_fit_time": "mean_fit_time_10",
        "mean_test_score": "mean_test_score_10",
        "t_not_significantly_worse": "t_not_significantly_worse_10",
        "t_values": "t_values_10",
    }
)

cv_renamed_100_pl = cv_results_100_pl.select(
    "param_model", "params", "mean_test_score", "rank_test_score", "mean_fit_time", "t_values", "t_not_significantly_worse"
).rename(
    {
        "rank_test_score": "rank_test_score_100",
        "mean_fit_time": "mean_fit_time_100",
        "mean_test_score": "mean_test_score_100",
        "t_not_significantly_worse": "t_not_significantly_worse_100",
        "t_values": "t_values_100",
    }
)

rank_compared_pl = (
    cv_renamed_1_pl
    .join(cv_renamed_10_pl, on=["param_model", "params"], how="inner")
    .join(cv_renamed_100_pl, on=["param_model", "params"], how="inner")
    .with_columns(
        (pl.col("rank_test_score_100") - pl.col("rank_test_score_1")).alias("rank_difference_1"),
        (pl.col("rank_test_score_100") - pl.col("rank_test_score_10")).alias("rank_difference_10")
    )
)
display_wide(rank_compared_pl)

In [ ]:
sns.scatterplot(rank_compared_pl, x="mean_test_score_1", y="mean_test_score_10", hue="param_model", style="param_model")
plt.title("Test scores, 1% vs 10% populations")
plt.xlabel("F1 macro scores over 1% subset")
plt.ylabel("F1 macro scores over 10% subset")
plt.legend(title="Model", bbox_to_anchor=(1.02,1))
plt.show()
sns.scatterplot(rank_compared_pl, x="mean_test_score_1", y="mean_test_score_100", hue="param_model", style="param_model")
plt.title("Test scores, 1% vs 100% populations")
plt.xlabel("F1 macro scores over 1% subset")
plt.ylabel("F1 macro scores over 100% subset")
plt.legend(title="Model", bbox_to_anchor=(1.02,1))
plt.show()
sns.scatterplot(rank_compared_pl, x="mean_test_score_10", y="mean_test_score_100", hue="param_model", style="param_model")
plt.title("Test scores, 10% vs 100% populations")
plt.xlabel("F1 macro scores over 10% subset")
plt.ylabel("F1 macro scores over 100% subset")
plt.legend(title="Model", bbox_to_anchor=(1.02,1))
plt.show()

- Visually there is a an overall good correlation between F1 macro scores over the 1% population and 10% population.
- Some correlation between 1% and 100% populations but not perfect
- Good correlation between 10% and 100% populations

In [ ]:
display("1% vs 100%")
rank_compared_pl.select(pl.corr("mean_test_score_1", "mean_test_score_100", method="pearson").alias("pearson")).pipe(display)
rank_compared_pl.select(pl.corr("rank_test_score_1", "rank_test_score_100", method="spearman").alias("spearman")).pipe(display)
display("10% vs 100%")
rank_compared_pl.select(pl.corr("mean_test_score_10", "mean_test_score_100", method="pearson").alias("pearson")).pipe(display)
rank_compared_pl.select(pl.corr("rank_test_score_10", "rank_test_score_100", method="spearman").alias("spearman")).pipe(display)

- The pearson correlation between the scores over the 1% population and 100% population was fairly high 0.971. The spearman correlation of the ranks was 0.936. So the results for the 1% population is quite high but not perfect.
- The pearson correlation between the scores over the 10% population and 100% population was very high 0.998. The spearman correlation of the ranks was 0.927.

It's not possible to test all potential models and hyperparameters against the 10% or 100% populations due to time and resource constraints. An initial filter using 1% will be used to narrow down models initially, but the final candidates should be tested against the 10% and 100% population.


In [ ]:
rank_compared_pl.pipe(display_wide, rows=1000)

In [ ]:
rank_compared_pl.filter(pl.col("t_not_significantly_worse_1").ne(pl.col("t_not_significantly_worse_100")))

Most models that performed well on the 1% population also did well over the 100% population, but there were a couple of discrepencies. 
- Using the paired t test, models that were not significantly worse over the 1% population were the same at the 100% level. 



In [ ]:
rank_compared_pl.filter(pl.col("rank_test_score_10").ne(pl.col("rank_test_score_100")))

In [ ]:
rank_compared_pl.filter(pl.col("rank_test_score_1").ne(pl.col("rank_test_score_100")))


- Difference in rankings between 1%/10% and 100% population were between 0-3 ranks.

# 4. Test Various Models and Hyperparameters 

## 4.1 High level search

In [ ]:
vectorizer_tfidf = FeatureUnion(
    [
        (
            "word",
            TfidfVectorizer(
                analyzer="word",
                ngram_range=(1, 3),
                lowercase=True,
                min_df=1,
                norm=None,
                use_idf=True,
            ),
        ),
    ]
)

pipe = Pipeline([
    ("features", vectorizer_tfidf), 
    ("normalize", Normalizer(norm="l2")),
    ("model", DummyClassifier(random_state=SEED))])


param_grid = [
    {
        "model": [DummyClassifier()],
        "model__random_state": [SEED],
        "model__strategy": ["stratified", "most_frequent", "uniform"],
    },
    {
        "model": [SVC()],
        "model__kernel": ["linear"],
        "model__C": loguniform(1e-3, 1e3),
        "model__class_weight": [None, "balanced"],
    },
    {
        "model": [SVC()],
        "model__kernel": ["rbf"],
        "model__C": loguniform(1e-3, 1e3),
        "model__gamma": loguniform(1e-6, 1e-1),
        "model__class_weight": [None, "balanced"],
    },
    {
        "model": [SVC()],
        "model__kernel": ["poly"],
        "model__C": loguniform(1e-3, 1e3),
        "model__degree": [2,3],
        "model__gamma": loguniform(1e-6, 1e-2),
        "model__coef0":[0, 1],
        "model__class_weight": [None, "balanced"],
    },
    {
        "model": [SVC()],
        "model__kernel": ["sigmoid"],
        "model__C": loguniform(1e-3, 1e3),
        "model__gamma": loguniform(1e-6, 1e-1),
        "model__coef0":[0, 1],
        "model__class_weight": [None, "balanced"],
    },
    {
        "model": [LinearSVC()],
        "model__C": loguniform(1e-4, 1e4),
        "model__loss": ["hinge"],
        "model__dual": [True],
        "model__penalty": ["l2"],
        "model__class_weight": [None, "balanced"],
        "model__max_iter": [5000]
    },
    {
        "model": [LinearSVC()],
        "model__C": loguniform(1e-4, 1e4),
        "model__loss": ["squared_hinge"],
        "model__dual": [False],
        "model__penalty": ["l1"],
        "model__class_weight": [None, "balanced"],
        "model__max_iter": [5000]
    },
    {
        "model": [SGDClassifier()],
        "model__random_state": [SEED],
        "model__loss": ["hinge", "log_loss"],
        "model__penalty": ["l1", "l2"],
        "model__alpha": loguniform(1e-6, 1e-3),
        "model__class_weight": [None, "balanced"],
        "model__tol": [1e-3, 1e-4],
        "model__max_iter":[5000],
    },
    {
        "model": [SGDClassifier()],
        "model__random_state": [SEED],
        "model__loss": ["hinge", "log_loss"],
        "model__penalty": ["elasticnet"],
        "model__l1_ratio": np.linspace(0.05, 0.95, 10),
        "model__alpha": loguniform(1e-6, 1e-3),
        "model__class_weight": [None, "balanced"],
        "model__tol": [1e-3, 1e-4],
        "model__max_iter":[5000],
    },
    {
        "model": [DecisionTreeClassifier()],
        "model__random_state": [SEED],
        "model__class_weight": [None, "balanced"],
        "model__max_depth": [None, 10, 30, 60],
    },
    {
        "model": [RandomForestClassifier()],
        "model__random_state": [SEED],
        "model__class_weight": [None, "balanced"],
        "model__n_estimators": [200, 500, 1000],
        "model__max_depth": [None, 10, 20 , 40],
        "model__max_features": ["sqrt", "log2", 0.2, 0.5],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 5],
    },
    {
        "model": [MultinomialNB(), ComplementNB()],
        "model__alpha": loguniform(1e-6, 1.0),
    },
    {
        "model": [PassiveAggressiveClassifier()],
        "model__random_state": [SEED],
        "model__loss": ["hinge", "squared_hinge"],
        "model__C": loguniform(1e-5, 1e2),
        "model__average": [True, False],
        "model__class_weight": [None, "balanced"],
        "model__tol": [1e-3, 1e-4],
        "model__max_iter":[2000],
    },
]


In [ ]:
subset = DataSample.sample_10_pct
run_name = f"initial_search_hs_10000_v2{dataset_version}_{subset.label}"

In [ ]:
grid_search = HalvingRandomSearchCV(
    estimator=pipe, 
    param_distributions=param_grid, 
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED), 
    scoring="f1_macro", 
    n_jobs=-1,
    verbose=3,
    random_state=SEED,
    n_candidates=10000,
    factor=3,
    resource="n_samples",
    max_resources="auto",
    min_resources="exhaust"
    )

grid_search = run_grid_search(grid_search=grid_search, dataset_name=dataset_name, dataset_pl=dataset_pl, subset=subset, run_name=run_name)

In [ ]:
run, grid_search, cv_results_pl, image_paths = load_ml_flow(experiment_name=experiment_name, run_name=run_name)
cv_results_pl = compare_to_top(cv_results_pl)
display_wide(cv_results_pl.sort("rank_test_score", descending=False), rows=100)
display_group_params(cv_results_pl)

Refine hyperparameters based on outputs

## 4.2 Refined HalvingRandomSearchCV

In [ ]:
vectorizer_tfidf = TfidfVectorizer(
                analyzer="word",
                ngram_range=(1, 2),
                lowercase=True,
                min_df=1,
                norm="l2",
                use_idf=True,
            )


pipe = Pipeline([
    ("vectorizer", vectorizer_tfidf), 
    ("model", DummyClassifier(random_state=SEED))])


param_grid = [
    {
        "model": [DummyClassifier()],
        "model__random_state": [SEED],
        "model__strategy": ["stratified", "most_frequent", "uniform"],
    },
    {
        "model": [SVC()],
        "model__kernel": ["linear"],
        "model__C": loguniform(1e-3, 1e2),
        "model__class_weight": [None, "balanced"],
    },
    {
        "model": [SVC()],
        "model__kernel": ["rbf"],
        "model__C": loguniform(1e-3, 1e2),
        "model__gamma": loguniform(1e-6, 1e-1),
        "model__class_weight": [None, "balanced"],
    },
    {
        "model": [SVC()],
        "model__kernel": ["poly"],
        "model__C": loguniform(1e-3, 1e2),
        "model__degree": [2,3],
        "model__gamma": loguniform(1e-6, 1e-2),
        "model__coef0":[0, 1],
        "model__class_weight": [None, "balanced"],
    },
    {
        "model": [SVC()],
        "model__kernel": ["sigmoid"],
        "model__C": loguniform(1e-3, 1e2),
        "model__gamma": loguniform(1e-6, 1e-1),
        "model__coef0":[0, 1],
        "model__class_weight": [None, "balanced"],
    },
    {
        "model": [LinearSVC()],
        "model__C": loguniform(1e-4, 1e2),
        "model__loss": ["hinge"],
        "model__dual": [True],
        "model__penalty": ["l2"],
        "model__class_weight": [None, "balanced"],
        "model__max_iter": [5000]
    },
    {
        "model": [LinearSVC()],
        "model__C": loguniform(1e-4, 1e2),
        "model__loss": ["squared_hinge"],
        "model__dual": [False],
        "model__penalty": ["l1"],
        "model__class_weight": [None, "balanced"],
        "model__max_iter": [5000]
    },
    {
        "model": [SGDClassifier()],
        "model__random_state": [SEED],
        "model__loss": ["hinge", "log_loss"],
        "model__penalty": ["l1", "l2"],
        "model__alpha": loguniform(1e-6, 1e-3),
        "model__class_weight": [None, "balanced"],
        "model__tol": [1e-3, 1e-4],
        "model__max_iter":[5000],
    },
    {
        "model": [SGDClassifier()],
        "model__random_state": [SEED],
        "model__loss": ["hinge", "log_loss"],
        "model__penalty": ["elasticnet"],
        "model__l1_ratio": np.linspace(0.05, 0.95, 10),
        "model__alpha": loguniform(1e-6, 1e-3),
        "model__class_weight": [None, "balanced"],
        "model__tol": [1e-3, 1e-4],
        "model__max_iter":[5000],
    },
    {
        "model": [DecisionTreeClassifier()],
        "model__random_state": [SEED],
        "model__class_weight": [None, "balanced"],
        "model__max_depth": [None, 10, 30, 60],
    },
    {
        "model": [RandomForestClassifier()],
        "model__random_state": [SEED],
        "model__class_weight": [None, "balanced"],
        "model__n_estimators": [200, 500, 1000],
        "model__max_depth": [None, 10, 20 , 40],
        "model__max_features": ["sqrt", "log2", 0.2, 0.5],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 5],
    },
    {
        "model": [MultinomialNB(), ComplementNB()],
        "model__alpha": loguniform(1e-6, 1.0),
    },
    {
        "model": [PassiveAggressiveClassifier()],
        "model__random_state": [SEED],
        "model__loss": ["hinge", "squared_hinge"],
        "model__C": loguniform(1e-5, 1e2),
        "model__average": [True, False],
        "model__class_weight": [None, "balanced"],
        "model__tol": [1e-3, 1e-4],
        "model__max_iter":[2000],
    },
]


In [ ]:
subset = DataSample.sample_10_pct
run_name = f"initial_search_hs_300_v1{dataset_version}_{subset.label}"

In [ ]:
grid_search = HalvingRandomSearchCV(
    estimator=pipe, 
    param_distributions=param_grid, 
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED), 
    scoring="f1_macro", 
    n_jobs=-1,
    verbose=3,
    random_state=SEED,
    n_candidates=300,
    factor=2,
    resource="n_samples",
    max_resources="auto",
    min_resources=15_000
    )

grid_search = run_grid_search(grid_search=grid_search, dataset_name=dataset_name, dataset_pl=dataset_pl, subset=subset, run_name=run_name)

In [ ]:
run, grid_search, cv_results_pl, image_paths = load_ml_flow(experiment_name=experiment_name, run_name=run_name)
cv_results_pl = (
    cv_results_pl
    .pipe(compare_to_top)
    .pipe(add_confidence_interval, confidence=0.95)
)
display_wide(cv_results_pl.sort("rank_test_score", descending=False), rows=100)
display_group_params(cv_results_pl)

In [ ]:
plot_scores_vs_training_time(cv_results_pl)

- Poor scoring models were quicker
- Models that took a long time did have better scores
- There is a sweet spot where some models had high scores and were quick

In [ ]:
cv_top_results_pl = cv_results_pl.sort("rank_test_score", descending=False).filter(pl.col("t_not_significantly_worse"), pl.col("iter") == pl.col("iter").max())
cv_top_results_pl.select("param_model").unique().pipe(display_wide, rows=200)
display_wide(cv_top_results_pl, rows=200)

- Top 3 models, LinearSVC, SVC(linear) and PassiveAggressiveClassifier

### 4.2.1 Plot hyperparameters vs scores and metrics

In [ ]:
score_column = "mean_test_score"
time_column = "mean_fit_time"
cv_params_pl = cv_results_pl.select(pl.col(r"^param_.*$"), score_column, time_column)
display(cv_params_pl)

param_columns = [col for col in cv_params_pl.columns]
param_models = cv_results_pl.select("param_model").unique().to_numpy()

def plot_param_vs_score(model_pl: pl.DataFrame, col: str, score_column: str, title_prefix: str="") -> None:
    x=model_pl[col]
    x_label=col
    if(x.dtype.is_float() and x.min() is not None and x.max() is not None and ((x.min() == 0) or (x.max()/x.min()) > 1000)):
         x = np.log(x)
         x_label=f"log {x_label}"
    
    if(x.dtype == pl.String):
        x = x.fill_null("None")
    sns.scatterplot(x=x, y=model_pl[score_column])
    plt.xlabel(x_label)
    plt.ylabel(score_column)
    plt.title(f"{title_prefix} {score_column} vs {col}")
    plt.show()


for grid_search in param_models:
    model_pl = cv_params_pl.filter(pl.col("param_model") == grid_search)
    good_models_pl = cv_results_pl.filter(pl.col("t_not_significantly_worse"), pl.col("param_model") == grid_search)
    for col in param_columns:
        if(model_pl[col].unique().len() > 1):
            plot_param_vs_score(model_pl, col, score_column, title_prefix=grid_search)
            plot_param_vs_score(good_models_pl, col, score_column, title_prefix=f"{grid_search} (t_not_significantly_worse)")


Models and hyperparams that look promising used in the next step

## 4.3 Three candidate models over the full testing poulation

In [ ]:
vectorizer_tfidf = TfidfVectorizer(
                analyzer="word",
                ngram_range=(1, 2),
                lowercase=True,
                min_df=1,
                norm="l2",
                use_idf=True,
            )


pipe = Pipeline([
    ("vectorizer", vectorizer_tfidf), 
    ("model", DummyClassifier(random_state=SEED))])


param_grid = [
    {
        "model": [SVC()],
        "model__kernel": ["linear"],
        "model__C": loguniform(1, 1e2),
        "model__class_weight": [None, "balanced"],
    },
    {
        "model": [LinearSVC()],
        "model__C": loguniform(1, 1e2),
        "model__loss": ["squared_hinge"],
        "model__dual": [False],
        "model__penalty": ["l1"],
        "model__class_weight": [None, "balanced"],
        "model__max_iter": [5000]
    },
    {
        "model": [PassiveAggressiveClassifier()],
        "model__random_state": [SEED],
        "model__loss": ["hinge", "squared_hinge"],
        "model__C": loguniform(1e-5, 1e2),
        "model__average": [True, False],
        "model__class_weight": [None, "balanced"],
        "model__tol": [1e-3, 1e-4],
        "model__max_iter":[2000],
    },
]


In [ ]:
subset = DataSample.sample_100_pct
run_name = f"candidate_models_hs_100_v3_{dataset_version}_{subset.label}"


In [ ]:
grid_search = HalvingRandomSearchCV(
    estimator=pipe, 
    param_distributions=param_grid, 
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED), 
    scoring="f1_macro", 
    n_jobs=-1,
    verbose=3,
    random_state=SEED,
    n_candidates=100,
    factor=2,
    resource="n_samples",
    max_resources="auto",
    min_resources=20_000,
    )

grid_search = run_grid_search(grid_search=grid_search, dataset_name=dataset_name, dataset_pl=dataset_pl, subset=subset, run_name=run_name)

In [ ]:
run, grid_search, cv_results_pl, image_paths = load_ml_flow(experiment_name=experiment_name, run_name=run_name)
cv_results_pl = (cv_results_pl
                 .pipe(compare_to_top)
                 .pipe(add_confidence_interval, confidence=0.95))
display_wide(cv_results_pl.sort("rank_test_score", descending=False), rows=100)
display_group_params(cv_results_pl)

- LinearSVC had the best score with 0.786538.
- C from  1-10 had good scores. 
- Balanced did better

- SVC had score of 0.773804, which was significantly worse at the 95% confidence level.

- PassiveAggressive has score of 0.755121, which was significantly worse at the 95% confidence level.

## 4.4 Candidate gridsearch.  
Use gridsearch to fully test hyperparameters


In [ ]:
vectorizer_tfidf = TfidfVectorizer(
                analyzer="word",
                ngram_range=(1, 2),
                lowercase=True,
                min_df=1,
                norm="l2",
                use_idf=True,
            )


pipe = Pipeline([
    ("vectorizer", vectorizer_tfidf), 
    ("model", DummyClassifier(random_state=SEED))])


param_grid = [
    {
        "model": [LinearSVC()],
        "model__C": np.arange(1, 10, 0.25),
        "model__loss": ["squared_hinge"],
        "model__dual": [False],
        "model__penalty": ["l1"],
        "model__class_weight": ["balanced"],
        "model__tol": [1e-3, 1e-4],
        "model__max_iter": [5000, 10000]
    },
]

In [ ]:
subset = DataSample.sample_1_pct
run_name = f"final_candidate_gs_{dataset_version}_{subset.label}"


In [ ]:
grid_search = GridSearchCV(
    estimator=pipe, 
    param_grid=param_grid, 
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED), 
    scoring="f1_macro", 
    n_jobs=-1,
    verbose=3)

grid_run = run_grid_search(grid_search=grid_search, dataset_name=dataset_name, dataset_pl=dataset_pl, subset=subset, run_name=run_name)



In [ ]:
run, grid_search, cv_results_pl, image_paths = load_ml_flow(experiment_name=experiment_name, run_name=run_name)
cv_results_pl = (cv_results_pl
                 .pipe(compare_to_top)
                 .pipe(add_confidence_interval, confidence=0.95))
display_wide(cv_results_pl.sort("rank_test_score", descending=False), rows=100)
display_group_params(cv_results_pl)

Top model was "{'model': LinearSVC(), 'model__C': 2.0, 'model__class_weight': 'balanced', 'model__dual': False, 'model__loss': 'squared_hinge', 'model__max_iter': 5000, 'model__penalty': 'l1', 'model__tol': 0.0001}"

- C didn't seem to have too much of an impact in the range tested. 
- tol of 0.0001 had the top 10 scores
- max iterations of 5000 had the top scores, so increasing it to 10,000, doesn't apear to significantly increase performance.

# 5 Optimise vectorisation with candidate model

## 5.1 TfidfVectorization

### 5.1.1 Word and Character vectorization

In [ ]:
vectorizer_tfidf = FeatureUnion(
    transformer_list=[
        (
            "word",
            TfidfVectorizer(
                analyzer="word",
                ngram_range=(1, 3),
                lowercase=True,
                min_df=1,
                norm=None,
                use_idf=True,
            ),
        ),
        (
            "char",
            TfidfVectorizer(
                analyzer="char_wb",
                ngram_range=(3, 4),
                lowercase=True,
                min_df=1,
                norm="l2",
                use_idf=False,
            ),
        ),
    ],
    transformer_weights={"word": 1.0, "char": 1.0},
)

pipe = Pipeline(
    [
        ("features", vectorizer_tfidf),
        ("normalize", Normalizer(norm="l2")),
        (
            "model",
            LinearSVC(
                random_state=SEED,
                penalty="l1",
                C=2.0,
                loss="squared_hinge",
                dual=False,
                class_weight="balanced",
                max_iter=5000,
                tol=1e-4,
            ),
        ),
    ]
)

param_grid = {
    "model__C": np.arange(0.5, 4, 0.25),
    "features__word__ngram_range": [(1, 2), (1, 3), (1, 4)],
    "features__word__min_df": [1, 2],
    "features__word__max_df": [0.9, 0.95, 1.0],
    "features__word__use_idf": [True, False],
    "features__word__norm": [None, "l2"],
    "features__word__sublinear_tf": [True, False],
    "features__char__ngram_range": [(3, 4), (3, 5), (3, 6), (4, 6)],
    "features__char__min_df": [1, 2],
    "features__char__max_df": [0.9, 0.95, 1.0],
    "features__char__use_idf": [True, False],
    "features__char__norm": [None, "l2"],
    "features__char__sublinear_tf": [True, False],
    "normalize": ["passthrough", Normalizer(norm="l2")],
    "features__transformer_weights": [
        {"word": 1.0, "char": 1.0},
        {"word": 2.0, "char": 1.0},
        {"word": 1.0, "char": 2.0},
    ],
}


In [ ]:
# There are lots of candidates and some tfidf configs are very slow, so use just 10% to narrow down range before testing on larger subsets
subset = DataSample.sample_10_pct
run_name = f"tfidf_hs_1000_v2_{dataset_version}_{subset.label}"

In [ ]:


grid_search = HalvingRandomSearchCV(
    estimator=pipe, 
    param_distributions=param_grid, 
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED), 
    scoring="f1_macro", 
    n_jobs=-1,
    verbose=3,
    random_state=SEED,
    n_candidates=1000,
    factor=2,
    resource="n_samples",
    max_resources="auto",
    # There are lots of candidates and some tfidf configs are very slow, so use this to narrow down range
    min_resources="exhaust"
    )

grid_search = run_grid_search(grid_search=grid_search, dataset_name=dataset_name, dataset_pl=dataset_pl, subset=subset, run_name=run_name)

In [ ]:
run, grid_search, cv_results_pl, image_paths = load_ml_flow(experiment_name=experiment_name, run_name=run_name)
cv_results_pl = compare_to_top(cv_results_pl)
display_wide(cv_results_pl.sort("rank_test_score", descending=False), rows=200)

In [ ]:
last_iter_pl = cv_results_pl.filter(pl.col("iter") == pl.col("iter").max())

for param in last_iter_pl.select(pl.col(r"^param_.*$")).columns:
    print(f"Param {param}:")
    display(last_iter_pl.select(pl.col(param)).group_by(param).count())



#### 5.1.1.1 Plot hyperparameters vs scores

In [ ]:
cv_params_pl

In [ ]:
score_column = "mean_test_score"
time_column = "mean_fit_time"
cv_params_pl = cv_results_pl.select(pl.col(r"^param_.*$"), score_column, time_column)
display(cv_params_pl)

param_columns = [col for col in cv_params_pl.columns]

cv_results_filtered_pl = cv_results_pl.filter(pl.col("iter") == pl.col("iter").max())

for col in param_columns:
    plot_param_vs_score(cv_results_filtered_pl, col, score_column, title_prefix="LinearSVC")



Top ranking "{'normalize': Normalizer(), 'model__C': 3.0, 'features__word__use_idf': True, 'features__word__sublinear_tf': True, 'features__word__norm': 'l2', 'features__word__ngram_range': (1, 3), 'features__word__min_df': 1, 'features__word__max_df': 1.0, 'features__transformer_weights': {'word': 2.0, 'char': 1.0}, 'features__char__use_idf': True, 'features__char__sublinear_tf': False, 'features__char__norm': 'l2', 'features__char__ngram_range': (3, 4), 'features__char__min_df': 2, 'features__char__max_df': 0.9}"

- It weights the words more highly than characters. 
- It uses idf over the characters, so there might be very common sub characters that are ranked more lowerly.
- It doesn't use idf over the words, so even more common words have important meaning.
- The differences between various options seems to be fairly small. It might be worth trying simplier vectorization to see how they compare.
- Of the latest itteration the best score wasn't shown to be significantly better at a 95% confidence level, and they all had similar/close scores  
- Most word ngrams were (1,3) so use that
- word max_df 0.95-1
- word idf true
- model C 3 best but also 2, 2.5, 1.75 did well
- Weightings were varied
- character idf mixed
- character sublinear best ones were false but still a few true
- character most norms l2, but one null
- character ngram range (4,6), (3,4), (3,6), (3,5)
- character min_df 2 was best but rest were mixed
- character max_df 0.9 and 0.95



### 5.1.2 Simplified vectorization
The optimum vectorisation is quite complicated and hyperparameter combinations aren't very easily undestood, compare them to simplier vectorisation


In [ ]:
vectorizer_tfidf = FeatureUnion(
    transformer_list=[
        (
            "word",
            TfidfVectorizer(
                analyzer="word",
                ngram_range=(1, 3),
                lowercase=True,
                min_df=1,
                norm=None,
                use_idf=True,
            ),
        ),
        (
            "char",
            TfidfVectorizer(
                analyzer="char_wb",
                ngram_range=(3, 4),
                lowercase=True,
                min_df=1,
                norm="l2",
                use_idf=False,
            ),
        ),
    ],
    transformer_weights={"word": 1.0, "char": 1.0},
)

pipe = Pipeline(
    [
        ("features", vectorizer_tfidf),
        (
            "model",
            LinearSVC(
                random_state=SEED,
                penalty="l1",
                C=3.0,
                loss="squared_hinge",
                dual=False,
                class_weight="balanced",
                max_iter=10000,
            ),
        ),
    ]
)

param_grid = [
    # Best params from previous search
    {
        "features__word__ngram_range": [(1, 3)],
        "features__word__min_df": [1],
        "features__word__max_df": [1.0],
        "features__word__use_idf": [True],
        "features__word__norm": ["l2"],
        "features__word__sublinear_tf": [True],
        "features__char__ngram_range": [(3, 4)],
        "features__char__min_df": [2],
        "features__char__max_df": [0.9],
        "features__char__use_idf": [True],
        "features__char__norm": ["l2"],
        "features__char__sublinear_tf": [False],

        "features__transformer_weights": [
            {"word": 2.0, "char": 1.0},
        ],
    },
    # Just words
        {
        "features__word__ngram_range": [(1, 2), (1, 3), (1, 4)],
        "features__word__min_df": [1],
        "features__word__max_df": [1.0],
        "features__word__use_idf": [True],
        "features__word__norm": ["l2"],
        "features__word__sublinear_tf": [True],
        "features__char": ["drop"],
    },

]


In [ ]:
subset = DataSample.sample_10_pct

run_name = f"Test_simplified_vectorization_gs{dataset_version}_{subset.label}"

grid_search = GridSearchCV(
    estimator=pipe, 
    param_grid=param_grid, 
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED), 
    scoring="f1_macro", 
    n_jobs=-1,
    verbose=3,
    )

grid_search = run_grid_search(grid_search=grid_search, dataset_name=dataset_name, dataset_pl=dataset_pl, subset=subset, run_name=run_name)

In [ ]:
run, grid_search, cv_results_pl, image_paths = load_ml_flow(experiment_name=experiment_name, run_name=run_name)
cv_results_pl = compare_to_top(cv_results_pl)
display_wide(cv_results_pl.sort("rank_test_score", descending=False), rows=200)

In [ ]:
subset = DataSample.sample_100_pct
run_name = f"compare_datasets_v4_{dataset_version}_{subset.label}"
grid_search = GridSearchCV(
    estimator=pipe, 
    param_grid=param_grid, 
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED), 
    scoring="f1_macro", 
    n_jobs=-1,
    verbose=3)

run_grid_search(grid_search=grid_search, dataset_name=dataset_name, dataset_pl=dataset_pl, subset=subset, run_name=run_name)

In [ ]:
run, grid_search, cv_results_pl, image_paths = load_ml_flow(experiment_name=experiment_name, run_name=run_name)
cv_results_pl = compare_to_top(cv_results_pl)
display_wide(cv_results_pl.sort("rank_test_score", descending=False))


The best model was with score of 0.790248, took 11209s "{'features__char__max_df': 0.9, 'features__char__min_df': 2, 'features__char__ngram_range': (3, 4), 'features__char__norm': 'l2', 'features__char__sublinear_tf': False, 'features__char__use_idf': True, 'features__transformer_weights': {'word': 2.0, 'char': 1.0}, 'features__word__max_df': 1.0, 'features__word__min_df': 1, 'features__word__ngram_range': (1, 3), 'features__word__norm': 'l2', 'features__word__sublinear_tf': True, 'features__word__use_idf': True}"

At a 95% confidence level it was better than the other models, but in absolute terms it wasn't by much. 

With just a word only model with a score of 0.786864 took 4554s "{'features__char': 'drop', 'features__word__max_df': 1.0, 'features__word__min_df': 1, 'features__word__ngram_range': (1, 4), 'features__word__norm': 'l2', 'features__word__sublinear_tf': True, 'features__word__use_idf': True}"

So a difference of 0.003384 but it took 2.46 times as long. 

- A word only ngram is more easily explainable
- Much faster
- Easier to understand 



## 5.2 Sentence embeddings

In [ ]:
model_e5 = SentenceTransformer("intfloat/e5-base-v2", device="mps")
model_sbert = SentenceTransformer("all-mpnet-base-v2", device="mps")

Testing this on the 1% population should be fine, it's just about embedding, just want a rough idea of any difference, don't need to be very specific. 

In [ ]:
vectorizer_tfidf = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 4),
    lowercase=True,
    min_df=1,
    max_df=1.0,
    norm="l2",
    use_idf=True,
    # This probably does nothing, but don't want to switch out randomly.
    sublinear_tf=True,
)

vectorizer_sbert = SentenceVectorizer(model_name="all-mpnet-base-v2")
vectorizer_e5 = SentenceVectorizer(model_name="intfloat/e5-base-v2", prefix="passage: ")

pipe = Pipeline(
    steps=[
        ("vectorizer", vectorizer_tfidf),
        (
            "model",
            LinearSVC(
                random_state=SEED,
                penalty="l1",
                C=3.0,
                loss="squared_hinge",
                dual=False,
                class_weight="balanced",
                max_iter=10000,
            ),
        ),
    ],
)


param_grid = model_grid = [
    {
        "vectorizer": [vectorizer_tfidf, vectorizer_sbert, vectorizer_e5],
    },
]

In [ ]:
subset = DataSample.sample_1_pct
run_name = f"test2_embeddings_v2_{dataset_version}_{subset.label}"

In [ ]:
grid_search = GridSearchCV(
    estimator=pipe, 
    param_grid=param_grid, 
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED), 
    scoring="f1_macro", 
    n_jobs=-1,
    verbose=3,
    )

grid_search = run_grid_search(grid_search=grid_search, dataset_name=dataset_name, dataset_pl=dataset_pl, subset=subset, run_name=run_name, save_grid=False)

In [ ]:
run, grid_search, cv_results_pl, image_paths = load_ml_flow(experiment_name=experiment_name, run_name=run_name, load_grid_search=False)
cv_results_pl = compare_to_top(cv_results_pl)
display_wide(cv_results_pl)

In [ ]:
subset = DataSample.sample_10_pct
run_name = f"test_embeddings_{dataset_version}_{subset.label}"

In [ ]:
grid_search = GridSearchCV(
    estimator=pipe, 
    param_grid=param_grid, 
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED), 
    scoring="f1_macro", 
    n_jobs=-1,
    verbose=3,
    )

grid_search = run_grid_search(grid_search=grid_search, dataset_name=dataset_name, dataset_pl=dataset_pl, subset=subset, run_name=run_name, save_grid=False)

In [ ]:
run, grid_search, cv_results_pl, image_paths = load_ml_flow(
    experiment_name=experiment_name, run_name=run_name
)
cv_results_pl = cv_results_pl.pipe(compare_to_top).pipe(
    add_confidence_interval, confidence=0.95
)
display_wide(cv_results_pl)

- mpnet had the best scores at 0.779311 and took 9,602s to fit
- e5 had the second best score at 0.778137, and wasn't significantly worse at the 95% confidence level and took 9,938s to fit.
- tfidf scored worse at 0.776054 and was significantly worse at the 95% confidence level. But it took only 143s to fit.

In absolute terms tfidf wasn't that much worse, only 0.003 worse f1 score but was 67 times faster.   
It would be best to go with the simpler, more explainable and faster TfidfVectorizer


# 6. Final candidate and tfidf testing

Do a grid search with small variations



In [ ]:
vectorizer_tfidf = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 3),
    lowercase=True,
    min_df=1,
    norm=None,
    use_idf=True,
)

pipe = Pipeline(
    [
        ("vectorizer", vectorizer_tfidf),
        (
            "model",
            LinearSVC(
                random_state=SEED,
                penalty="l1",
                C=3.0,
                loss="squared_hinge",
                dual=False,
                class_weight="balanced",
                max_iter=10000,
            ),
        ),
    ]
)

param_grid = [
    {
        "model__C": np.arange(2.5, 3.5, 0.1),
        "model__max_iter": [5000, 10000, 20000],
        "vectorizer__ngram_range": [(1, 2), (1, 3), (1, 4)],
        "vectorizer__min_df": [1, 2],
        "vectorizer__max_df": [0.95, 1.0],
        "vectorizer__use_idf": [True, False],
        "vectorizer__norm": ["l2"],
        "vectorizer__sublinear_tf": [True, False],
        
    },
]

In [ ]:
subset = DataSample.sample_10_pct
run_name = f"Final_candidate_hyperparameters_{dataset_version}_{subset.label}"

In [ ]:
grid_search = HalvingRandomSearchCV(
    estimator=pipe, 
    param_distributions=param_grid, 
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED), 
    scoring="f1_macro", 
    n_jobs=-1,
    verbose=2,
    random_state=SEED,
    n_candidates=4320,
    factor=2,
    resource="n_samples",
    max_resources="auto",
    min_resources="exhaust"
    )

grid_search = run_grid_search(grid_search=grid_search, dataset_name=dataset_name, dataset_pl=dataset_pl, subset=subset, run_name=run_name)

In [ ]:
run, grid_search, cv_results_pl, image_paths = load_ml_flow(experiment_name=experiment_name, run_name=run_name)
cv_results_pl = compare_to_top(cv_results_pl).pipe(add_confidence_interval, confidence=0.95)
display_wide(cv_results_pl.sort("rank_test_score", descending=False), rows=100)

In [ ]:
cv_results_pl.filter(pl.col("iter")==pl.col("iter").max()).sort(pl.col("rank_test_score"), descending=False).pipe(display_wide, rows=100)

In [ ]:
score_column = "mean_test_score"
time_column = "mean_fit_time"
cv_params_pl = cv_results_pl.select(pl.col(r"^param_.*$"), score_column, time_column)
display(cv_params_pl)

param_columns = [col for col in cv_params_pl.columns]
# param_models = cv_results_pl.select("param_model").unique().to_numpy()

cv_results_filtered_pl = cv_results_pl.filter(pl.col("iter") == pl.col("iter").max())

iters = cv_results_pl.select("iter").sort("iter").unique().to_numpy()

for iter in reversed(iters):
    print(f"{iter=}")
    cv_results_filtered_pl = cv_results_pl.filter(pl.col("iter") == iter)
    for col in param_columns:
        plot_param_vs_score(cv_results_filtered_pl, col, score_column, title_prefix="LinearSVC")

In [ ]:
cv_results_filtered_pl = cv_results_pl.filter(pl.col("iter") == 0)
col = "mean_test_score"
sns.scatterplot(data=cv_results_filtered_pl, x=col, y=time_column, hue="param_vectorizer__min_df")
plt.title(f"Two groups of models are split by min_df")
plt.show()

- Really interestingly having a min_df of 1 is faster and has a better score than min_df of 2. So that additional information makes it easier to fit, without the data it has to work longer and hard to try and find a fit but does bad at it
- Larger C has a slight increase in fit times which makes sense. Top scores were 2.5, and range was 2.5-3.0.
- 20k iterations was slightly better but not by much and it was slower.
- No clear difference in max_df
- ngrame range, (1,2) was clearly slower which is interesting, it must mean it requires more work using the less data. (1,3) had the best scores and all of the last itteration were (1,3)
- sublinear False was a touch better, covered most of the last itteration
- Idf true, covered most of the last itteration. 
- max_df 0.95 or 1 didn't matter. Which makes sense since it's a short phrase and words aren't likely doubled
- max_iter varied between 5k and 20k, no significant difference in the latest interration. But 20k is slower. Can select 10k as a nice balance, also top scores were 10k.

# 7 Final candidate model test scores
Test against all the different test populations to allow comparisions to other model types.  
Using gridsearch just so existing functions work over it

In [ ]:
vectorizer_tfidf = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 3),
    lowercase=True,
    min_df=1,
    norm="l2",
    use_idf=True,
)

pipe = Pipeline(
    [
        ("vectorizer", vectorizer_tfidf),
        (
            "model",
            LinearSVC(
                random_state=SEED,
                penalty="l1",
                C=2.8,
                loss="squared_hinge",
                dual=False,
                class_weight="balanced",
                max_iter=10000,
            ),
        ),
    ]
)

param_grid = model_grid = [
    
    {
    },
]


## 7.1 1% train population

In [ ]:
subset = DataSample.sample_1_pct
run_name = f"candidate_model_with_candidate_vectorizers_v2{dataset_version}_{subset.label}"

In [ ]:
grid_search = GridSearchCV(
    estimator=pipe, 
    param_grid=param_grid, 
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED), 
    scoring={"f1_macro": "f1_macro",
             "precision_macro": "precision_macro",
             "recall_macro": "recall_macro",
             "accuracy": "accuracy"}, 
    refit="f1_macro",
    n_jobs=-1,
    verbose=3,
    return_train_score=False,
    )

grid_search = run_grid_search(grid_search=grid_search, dataset_name=dataset_name, dataset_pl=dataset_pl, subset=subset, run_name=run_name)

In [ ]:
run, grid_search, cv_results_pl, image_paths = load_ml_flow(experiment_name=experiment_name, run_name=run_name, load_grid_search=True)
cv_results_pl = (
    cv_results_pl
    .pipe(compare_to_top)
    .pipe(add_confidence_interval, confidence=0.95)
    .pipe(add_confidence_interval, confidence=0.95, metric="accuracy")
)
display_wide(cv_results_pl)

In [ ]:
bootstrap_ci_fast(dataset_pl=dataset_pl, model=grid_search, test_field="test_5_pct", n_bootstrap=1000, ci=0.95)

In [ ]:
bootstrap_ci_fast(dataset_pl=dataset_pl, model=grid_search, test_field="test", n_bootstrap=1000, ci=0.95)

In [ ]:
bootstrap_ci_fast(dataset_pl=dataset_pl, model=grid_search, test_field="holdout_10k", n_bootstrap=1000, ci=0.95)

In [ ]:
bootstrap_ci_fast(dataset_pl=dataset_pl, model=grid_search, test_field="holdout", n_bootstrap=1000, ci=0.95)

1% test population.  
- Accuracy: 0.9717 (95% CI: 0.9710–0.9723) 
- F1 score: 0.7490 (95% CI: 0.7445–0.7527)

In [ ]:
run, grid_search, cv_results_pl, image_paths = load_ml_flow(experiment_name=experiment_name, run_name=run_name)
cv_results_pl = compare_to_top(cv_results_pl)
display_wide(cv_results_pl)

In [ ]:
def test_metrics(model: Pipeline, test_field: str) -> dict:

    test_pl = dataset_pl.filter(pl.col(test_field))
    predictions = model.predict(test_pl[X])
    classification = classification_report(predictions, test_pl[y], output_dict=True)
    accuracy = accuracy_score(test_pl[y], predictions)
    print(f"Accuracy: {accuracy}")
    print("Macro")
    print(classification["macro avg"])
    print("Weighted avg")
    print(classification["weighted avg"])

    return classification

test_metrics(model=grid_search, test_field="holdout")

## 7.2 10% train population

In [ ]:
subset = DataSample.sample_10_pct
run_name = f"candidate_models_with_candidate_vectorizers_{dataset_version}_{subset.label}"

In [ ]:
grid_search = GridSearchCV(
    estimator=pipe, 
    param_grid=param_grid, 
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED), 
    scoring="f1_macro", 
    n_jobs=-1,
    verbose=3,
    )

grid_search = run_grid_search(grid_search=grid_search, dataset_name=dataset_name, dataset_pl=dataset_pl, subset=subset, run_name=run_name)

In [ ]:
run, grid_search, cv_results_pl, image_paths = load_ml_flow(experiment_name=experiment_name, run_name=run_name)
cv_results_pl = compare_to_top(cv_results_pl).pipe(add_confidence_interval, confidence=0.95)
display_wide(cv_results_pl)

In [ ]:
test_metrics(model=grid_search, test_field="holdout")

In [ ]:
bootstrap_ci_fast(dataset_pl=dataset_pl, model=grid_search, test_field="test_5_pct", n_bootstrap=1000, ci=0.95)

In [ ]:
bootstrap_ci_fast(dataset_pl=dataset_pl, model=grid_search, test_field="test", n_bootstrap=1000, ci=0.95)

In [ ]:
bootstrap_ci_fast(dataset_pl=dataset_pl, model=grid_search, test_field="holdout_10k", n_bootstrap=1000, ci=0.95)

In [ ]:
bootstrap_ci_fast(dataset_pl=dataset_pl, model=grid_search, test_field="holdout", n_bootstrap=1000, ci=0.95)

## 7.3 100% train population

In [ ]:
subset = DataSample.sample_100_pct
run_name = f"candidate_models_with_candidate_vectorizers_{dataset_version}_{subset.label}"

In [ ]:
grid_search = GridSearchCV(
    estimator=pipe, 
    param_grid=param_grid, 
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED), 
    scoring="f1_macro", 
    n_jobs=-1,
    verbose=3,
    )

grid_search = run_grid_search(grid_search=grid_search, dataset_name=dataset_name, dataset_pl=dataset_pl, subset=subset, run_name=run_name)

In [ ]:
run, grid_search, cv_results_pl, image_paths = load_ml_flow(experiment_name=experiment_name, run_name=run_name)
cv_results_pl = compare_to_top(cv_results_pl)
display_wide(cv_results_pl)

In [ ]:
bootstrap_ci_fast(dataset_pl=dataset_pl, model=grid_search, test_field="test_5_pct", n_bootstrap=1000, ci=0.95)

In [ ]:
bootstrap_ci_fast(dataset_pl=dataset_pl, model=grid_search, test_field="test", n_bootstrap=1000, ci=0.95)

In [ ]:
bootstrap_ci_fast(dataset_pl=dataset_pl, model=grid_search, test_field="holdout_10k", n_bootstrap=1000, ci=0.95)

In [ ]:
bootstrap_ci_fast(dataset_pl=dataset_pl, model=grid_search, test_field="holdout", n_bootstrap=1000, ci=0.95)

In [ ]:
test_metrics(model=grid_search, test_field="holdout")

## 7.4 1% sqrt weighted training population

### 7.4.1 Embeddings

In [ ]:
vectorizer_mpnet = SentenceVectorizer(model_name="all-mpnet-base-v2")
vectorizer_e5 = SentenceVectorizer(model_name="intfloat/e5-base-v2", prefix = "passage: ")

vectorizer_tfidf = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 3),
    lowercase=True,
    min_df=1,
    norm="l2",
    use_idf=True,
)

pipe = Pipeline(
    [
        ("vectorizer", vectorizer_tfidf),
        (
            "model",
            LinearSVC(
                random_state=SEED,
                penalty="l1",
                C=2.8,
                loss="squared_hinge",
                dual=False,
                class_weight="balanced",
                max_iter=10000,
            ),
        ),
    ]
)

param_grid = model_grid = [
    
    { "vectorizer": [vectorizer_tfidf, vectorizer_mpnet, vectorizer_e5],
    },
]


In [ ]:
subset = DataSample.sample_1_pct_sqrt_weight
run_name = f"final_candidate_model_{dataset_version}_{subset.label}"

In [ ]:
grid_search = GridSearchCV(
    estimator=pipe, 
    param_grid=param_grid, 
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED), 
    scoring="f1_macro", 
    n_jobs=-1,
    verbose=3,
    )

grid_search = run_grid_search(grid_search=grid_search, dataset_name=dataset_name, dataset_pl=dataset_pl, subset=subset, run_name=run_name)

In [ ]:
grid_search_backup = grid_search

In [ ]:
run, grid_search, cv_results_pl, image_paths = load_ml_flow(experiment_name=experiment_name, run_name=run_name, load_grid_search=False)
cv_results_pl = compare_to_top(cv_results_pl).pipe(add_confidence_interval, confidence=0.95)
display_wide(cv_results_pl)


- mpnet has the best score but only by a small amount, 0.77051
- tfidf came second with score of 0.770445, and wasn't significantly different than the top score at 95% confidence level
- e5 came last but wasn't significantly different than the top score at 95% confidence level

In [ ]:
grid_search = grid_search_backup

In [ ]:
pipe = grid_search.best_estimator_
test_metrics(model=pipe, test_field="holdout")

In [ ]:
bootstrap_ci_fast(dataset_pl=dataset_pl, model=grid_search, test_field="test_5_pct", n_bootstrap=1000, ci=0.95)

In [ ]:
bootstrap_ci_fast(dataset_pl=dataset_pl, model=grid_search, test_field="test", n_bootstrap=1000, ci=0.95)

In [ ]:
bootstrap_ci_fast(dataset_pl=dataset_pl, model=grid_search, test_field="holdout", n_bootstrap=1000, ci=0.95)

Using the sqrt weighted training dataset macro-f1 increased by 3 pp

### 7.4.2 tfidf vectorization scores


In [ ]:
subset = DataSample.sample_1_pct_sqrt_weight
run_name = f"final_model_{dataset_version}_{subset.label}"

In [ ]:
grid_search = GridSearchCV(
    estimator=pipe, 
    param_grid=param_grid, 
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED), 
    scoring={"f1_macro": "f1_macro",
             "precision_macro": "precision_macro",
             "recall_macro": "recall_macro",
             "accuracy": "accuracy"}, 
    refit="f1_macro",
    n_jobs=-1,
    verbose=3,
    return_train_score=False,
    )

grid_search = run_grid_search(grid_search=grid_search, dataset_name=dataset_name, dataset_pl=dataset_pl, subset=subset, run_name=run_name)

In [ ]:
run, grid_search, cv_results_pl, image_paths = load_ml_flow(experiment_name=experiment_name, run_name=run_name, load_grid_search=True)
cv_results_pl = compare_to_top(cv_results_pl).pipe(add_confidence_interval, confidence=0.95).pipe(add_confidence_interval, confidence=0.95, metric="accuracy")
display_wide(cv_results_pl)


In [ ]:
bootstrap_ci_fast(dataset_pl=dataset_pl, model=grid_search, test_field="test_5_pct", n_bootstrap=1000, ci=0.95)

In [ ]:
bootstrap_ci_fast(dataset_pl=dataset_pl, model=grid_search, test_field="test", n_bootstrap=1000, ci=0.95)

In [ ]:
bootstrap_ci_fast(dataset_pl=dataset_pl, model=grid_search, test_field="holdout_10k", n_bootstrap=1000, ci=0.95)

In [ ]:
bootstrap_ci_fast(dataset_pl=dataset_pl, model=grid_search, test_field="holdout", n_bootstrap=1000, ci=0.95)

- The f1 macro score is 2 pp higher using sqrt weighted 1% training dataset vs 1% training dataset. 

In [ ]:
pipe = grid_search.best_estimator_
test_metrics(model=pipe, test_field="holdout")

## 7.5 10% sqrt weighted training population

In [ ]:
subset = DataSample.sample_10_pct_sqrt_weight
run_name = f"final_model_{dataset_version}_{subset.label}"

In [ ]:
grid_search = GridSearchCV(
    estimator=pipe, 
    param_grid=param_grid, 
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED), 
    scoring={"f1_macro": "f1_macro",
             "precision_macro": "precision_macro",
             "recall_macro": "recall_macro",
             "accuracy": "accuracy"}, 
    refit="f1_macro",
    n_jobs=-1,
    verbose=3,
    return_train_score=False,
    )

grid_search = run_grid_search(grid_search=grid_search, dataset_name=dataset_name, dataset_pl=dataset_pl, subset=subset, run_name=run_name)

In [ ]:
run, grid_search, cv_results_pl, image_paths = load_ml_flow(experiment_name=experiment_name, run_name=run_name)

In [ ]:
bootstrap_ci_fast(dataset_pl=dataset_pl, model=grid_search, test_field="test_5_pct", n_bootstrap=1000, ci=0.95)

In [ ]:
bootstrap_ci_fast(dataset_pl=dataset_pl, model=grid_search, test_field="test", n_bootstrap=1000, ci=0.95)

In [ ]:
bootstrap_ci_fast(dataset_pl=dataset_pl, model=grid_search, test_field="holdout_10k", n_bootstrap=1000, ci=0.95)

In [ ]:
bootstrap_ci_fast(dataset_pl=dataset_pl, model=grid_search, test_field="holdout", n_bootstrap=1000, ci=0.95)

In [ ]:
pipe = grid_search.best_estimator_
test_metrics(model=pipe, test_field="holdout")

- F1 macro score is better than 10% training population but worse than 100% training population. 

# 8. Explainability what features does the model use to predict each class

In [ ]:
pipe = grid_search.best_estimator_

vectorizer = pipe["vectorizer"]
model = pipe["model"]

feature_names = vectorizer.get_feature_names_out()
classes_pl = dataset_pl.select("canonical_label", "label").unique()

for class_index, class_label in enumerate(model.classes_):
    display(classes_pl.filter(pl.col("label") == class_label))

    coef = model.coef_[class_index].ravel()

    print(f"Class: {class_index}")
    print(f"Label: {class_label}")
    print(f"Intercept: {model.intercept_[class_index]}")

    print("Positive:")
    print(feature_names[coef > 0])

    print("Negative:")
    print(feature_names[coef < 0])

The simply 1,3 word ngrams vectorization and LinearSVC helps provides clear explainability about what words(features) are used to classify descriptions 

In [ ]:
subset = DataSample.sample_1_pct

In [ ]:
from lime.lime_text import LimeTextExplainer

In [ ]:
class_names = [
    dataset_pl.select("canonical_label", y).unique().filter(pl.col("label") == label)["canonical_label"][0]
    for label in pipe.classes_
]

In [ ]:
run_name = 'candidate_models_with_candidate_vectorizers_v13_sample_100_pct'
run, grid_search, cv_results_pl, image_paths = load_ml_flow(experiment_name=experiment_name, run_name=run_name)

In [ ]:


mlflow.sklearn.autolog(disable=True)

explainer = LimeTextExplainer(class_names=class_names)

def lime_predict_proba(texts):
    scores = pipe.decision_function(texts)

    # Binary LinearSVC returns shape: (n_samples,)
    if scores.ndim == 1:
        positive = expit(scores)
        return np.column_stack([1 - positive, positive])

    # Multiclass returns shape: (n_samples, n_classes)
    return softmax(scores, axis=1)

exp = explainer.explain_instance(
    "cost of goods sold turnover",
    lime_predict_proba,
    top_labels=5,
)

html = exp.as_html()
display(HTML(f'<div style="background:white; padding:10px">{html}</div>'))

mlflow.sklearn.autolog(disable=False)

In [ ]:


mlflow.sklearn.autolog(disable=True)

pipe = grid_search.best_estimator_

train_pl, test_pl, _ = get_split(dataset_pl, subset=subset)

vectorizer = pipe.named_steps["vectorizer"]
model = pipe.named_steps["model"]

feature_names = vectorizer.get_feature_names_out()

train_vectorized = vectorizer.transform(train_pl[X]).toarray()
test_text = ["cost of goods sold turnover"]
test_vectorized = vectorizer.transform(test_text).toarray()

test_mask = test_vectorized[0] > 0
present_features = set(feature_names[test_mask])


def shap_predict_proba(vectorized_text):
    scores = model.decision_function(vectorized_text)

    # Binary LinearSVC returns shape: (n_samples,)
    if scores.ndim == 1:
        positive = expit(scores)
        return np.column_stack([1 - positive, positive])

    # Multiclass LinearSVC returns shape: (n_samples, n_classes)
    return softmax(scores, axis=1)


background = shap.sample(train_vectorized, 200, random_state=42)

explainer = shap.Explainer(
    shap_predict_proba,
    background,
    feature_names=feature_names,
)

shap_values = explainer(test_vectorized, max_evals=1600)

top_3 = (
    dataset_pl
    .filter(
        pl.col("canonical_label").is_in(
            ["TurnoverRevenue", "CostSales", "RawMaterialsConsumablesUsed"]
        )
    )
    .select("canonical_label", "label")
    .unique()
)

for row in top_3.iter_rows(named=True):
    class_idx = row["label"]
    class_name = row["canonical_label"]

    print(f"Class name: {class_name}")

    idx = np.flatnonzero(pipe.classes_ == class_idx)

    if len(idx) == 0:
        print(f"Skipping {class_name}: label {class_idx!r} not found in pipe.classes_")
        continue

    idx = idx[0]

    values = shap_values.values[0, :, idx].astype(float)

    present_feature_mask = np.array(
        [feature in present_features for feature in feature_names]
    )

    nonzero_mask = (np.abs(values) > 1e-10) & present_feature_mask

    nonzero_values = values[nonzero_mask]
    nonzero_names = feature_names[nonzero_mask]

    display_threshold = 5e-5  # anything that would show as ±0 at your axis scale

    display_mask = (np.abs(values) >= display_threshold) & present_feature_mask

    display_values = values[display_mask]
    display_names = feature_names[display_mask]

    if len(nonzero_values) == 0:
        print(f"No non-zero SHAP values for present features in {class_name}")
        continue

    shap.plots.bar(
        shap.Explanation(
            values=nonzero_values,
            feature_names=list(nonzero_names),
        ),
        max_display=len(nonzero_values),
        show=False,
    )

    ax = plt.gca()

    # remove ALL value labels (the numbers next to bars)
    for text in ax.texts:
        text.set_visible(False)

    plt.show()
    plt.close()

mlflow.sklearn.autolog(disable=False)

In [ ]:
coef_matrix = model.coef_  # shape: (n_classes, n_features)

for class_index, class_label in enumerate(model.classes_):

    canonical_label = (
        classes_pl
        .filter(pl.col("label") == class_label)
        .select("canonical_label")
        .item()
    )

    if canonical_label not in {
        "TurnoverRevenue",
        "CostSales",
        "RawMaterialsConsumablesUsed",
    }:
        continue

    coef = coef_matrix[class_index]

    print(f"\nClass: {canonical_label} | index: {class_index}")
    print(f"Intercept: {model.intercept_[class_index]}")

    pos_mask = coef > threshold
    neg_mask = coef < -threshold

    print("Positive:")
    print(feature_names[pos_mask])

    print("Negative:")
    print(feature_names[neg_mask])